# Notebook 01: Time-Series Preparation, EDA & Feature Engineering
**Project**: HHS Unaccompanied Alien Children (UAC) Program — Proactive Care Load & Discharge Demand Forecasting

### Objectives Covered in This Notebook
1. **Time-Series Preparation**:
   - Load `HHS_Unaccompanied_Alien_Children_Program.csv` and strip trailing blank rows.
   - Convert `Date` to a strict `datetime` index sorted chronologically (`2023-01-12` to `2025-12-21`).
   - Reindex to a continuous daily frequency (`1,075` days) and handle missing weekend/holiday reporting days via time-based interpolation and an `is_observed` binary mask.
2. **Classical Time-Series Decomposition**:
   - Decompose `Children in HHS Care` and `Children discharged from HHS Care` into **Trend**, **Weekly Seasonality**, and **Residuals**.
3. **Feature Engineering for Forecasting (72 Features)**:
   - Autoregressive lags (`t-1`, `t-2`, `t-3`, `t-7`, `t-14`).
   - Rolling 7-day and 14-day means and standard deviations (shifted to prevent data leakage).
   - Flow-based signals (`Transfers - Discharges` net pressure indicator, intake-to-exit ratio, CBP transfer efficiency).
   - Queueing-theory Implied Length of Stay (`LoS`) via Little's Law (`L = λW`).
   - Calendar and Fourier cyclical encodings.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

CSV_PATH = "../data/HHS_Unaccompanied_Alien_Children_Program.csv", "data/HHS_Unaccompanied_Alien_Children_Program.csv", "HHS_Unaccompanied_Alien_Children_Program.csv"

def load_and_prepare_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path).dropna(how="all").copy()
    df.columns = [c.strip() for c in df.columns]
    rename_map = {
        "Date": "date",
        "Children apprehended and placed in CBP custody*": "cbp_apprehended",
        "Children apprehended and placed in CBP custody": "cbp_apprehended",
        "Children in CBP custody": "cbp_custody",
        "Children transferred out of CBP custody": "transferred_to_hhs",
        "Children in HHS Care": "hhs_care",
        "Children discharged from HHS Care": "hhs_discharged",
    }
    df = df.rename(columns=rename_map)
    df["date"] = pd.to_datetime(df["date"], format="mixed")
    num_cols = ["cbp_apprehended", "cbp_custody", "transferred_to_hhs", "hhs_care", "hhs_discharged"]
    for col in num_cols:
        df[col] = df[col].astype(str).str.replace(",", "", regex=False).str.strip()
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values("date").drop_duplicates(subset=["date"], keep="last").set_index("date")
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq="D")
    df["is_observed"] = 1
    df = df.reindex(full_idx)
    df.index.name = "date"
    df["is_observed"] = df["is_observed"].fillna(0).astype(int)
    df[num_cols] = df[num_cols].interpolate(method="time").bfill().ffill()
    return df

df = load_and_prepare_data(CSV_PATH)
print(f"Continuous Date Range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Total Continuous Days: {len(df)} | Observed Reporting Days: {df['is_observed'].sum()} | Interpolated Days: {(df['is_observed'] == 0).sum()}")
df.describe().round(2)

Continuous Date Range: 2023-01-12 to 2025-12-21
Total Continuous Days: 1075 | Observed Reporting Days: 720 | Interpolated Days: 355
       cbp_apprehended  cbp_custody  transferred_to_hhs  hhs_care  hhs_discharged  is_observed
count          1075.00      1075.00             1075.00   1075.00         1075.00      1075.00
mean             92.01       169.10              126.96   6062.66          182.69         0.67
std              71.40       125.27               94.75   2826.98          131.34         0.47
min               0.00         7.00                0.00   1972.00            0.00         0.00
25%              12.00        35.00               14.00   2480.50           19.83         0.00
50%              99.67       188.00              152.00   6410.00          191.00         1.00
75%             144.50       259.17              196.00   8006.00          286.00         1.00
max             333.00       531.00              440.00  11516.00          505.00         1.00

In [2]:
# Step 2: Classical Time-Series Decomposition (Trend, Seasonality, Residuals)
def decompose_series(series: pd.Series, period: int = 7) -> pd.DataFrame:
    trend = series.rolling(window=period, center=True, min_periods=1).mean()
    detrended = series - trend
    seasonal = detrended.groupby(series.index.dayofweek).transform("mean")
    residual = series - trend - seasonal
    return pd.DataFrame({"Observed": series, "Trend": trend, "Seasonal": seasonal, "Residual": residual}, index=series.index)

decomp_care = decompose_series(df["hhs_care"], period=7)

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
decomp_care["Observed"].plot(ax=axes[0], color="#0f172a", title="Observed Children in HHS Care (2023-2025)")
decomp_care["Trend"].plot(ax=axes[1], color="#2563eb", title="7-Day Rolling Trend Component")
decomp_care["Seasonal"].iloc[-180:].plot(ax=axes[2], color="#0d9488", title="Weekly Day-of-Week Seasonality (Last 180 Days)")
decomp_care["Residual"].plot(ax=axes[3], color="#dc2626", title="Residuals & Structural Policy Shocks")
plt.tight_layout()
plt.show()

Displayed 4-panel seasonal decomposition figure for Children in HHS Care.

In [3]:
# Step 3: Feature Engineering (Lags, Rolling Stats, Flow Signals, Little's Law Implied LoS, Calendar Effects)
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    feat = df.copy()
    feat["net_flow"] = feat["transferred_to_hhs"] - feat["hhs_discharged"]
    feat["actual_care_change"] = feat["hhs_care"].diff().fillna(0)
    feat["intake_to_exit_ratio"] = (feat["transferred_to_hhs"] + 1.0) / (feat["hhs_discharged"] + 1.0)
    feat["cbp_transfer_efficiency"] = (feat["transferred_to_hhs"] + 1.0) / (feat["cbp_custody"] + 1.0)

    # Little's Law: Implied Length of Stay (Days) = Active HHS Care / 7-day rolling discharge rate
    rolling_disc_7 = feat["hhs_discharged"].rolling(7, min_periods=1).mean().clip(lower=1.0)
    feat["implied_los_days"] = feat["hhs_care"] / rolling_disc_7

    for col in ["hhs_care", "hhs_discharged", "transferred_to_hhs", "cbp_custody", "cbp_apprehended", "net_flow"]:
        for lag in [1, 2, 3, 7, 14]:
            feat[f"{col}_lag_{lag}"] = feat[col].shift(lag)
        for win in [7, 14]:
            feat[f"{col}_roll_mean_{win}"] = feat[col].shift(1).rolling(win).mean()
            feat[f"{col}_roll_std_{win}"] = feat[col].shift(1).rolling(win).std()

    feat["care_momentum_7d"] = feat["hhs_care"].shift(1) - feat["hhs_care"].shift(8)
    feat["care_momentum_14d"] = feat["hhs_care"].shift(1) - feat["hhs_care"].shift(15)
    feat["dayofweek"] = feat.index.dayofweek
    feat["month"] = feat.index.month
    feat["is_weekend"] = (feat["dayofweek"] >= 5).astype(int)
    feat["sin_dow"] = np.sin(2 * np.pi * feat["dayofweek"] / 7.0)
    feat["cos_dow"] = np.cos(2 * np.pi * feat["dayofweek"] / 7.0)
    return feat.dropna()

feat_df = engineer_features(df)
print(f"Engineered Feature Matrix Shape: {feat_df.shape}")
print("Sample Engineered Columns:", list(feat_df.columns[:18]))

Engineered Feature Matrix Shape: (1060, 72)
Sample Engineered Columns: ['cbp_apprehended', 'cbp_custody', 'transferred_to_hhs', 'hhs_care', 'hhs_discharged', 'is_observed', 'net_flow', 'actual_care_change', 'intake_to_exit_ratio', 'cbp_transfer_efficiency', 'implied_los_days', 'hhs_care_lag_1', 'hhs_care_lag_2', 'hhs_care_lag_3', 'hhs_care_lag_7', 'hhs_care_lag_14', 'hhs_care_roll_mean_7', 'hhs_care_roll_std_7']